# 33 - PF-026 a PF-030 - Frontend product UI

Genera el frontend final de producto para consumir el backend Spring Boot, con worklist, detalle, evidencia visual, estados de revisión y formulario de observaciones.

**Alcance:** no diagnostica; presenta evidencia técnica y apoyo a revisión profesional.

In [ ]:

from pathlib import Path
import json, textwrap, datetime
import pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print("No Colab drive mount:", e)

PFI_ROOT = Path("/content/drive/MyDrive/PFI_MVP")
REPO_ROOT = PFI_ROOT / "repo"
FRONTEND_ROOT = REPO_ROOT / "frontend_product_ui"
RESULTS_ROOT = PFI_ROOT / "results" / "PF026_PF030_frontend_product_ui"
DOCS_ROOT = REPO_ROOT / "docs"
BACKLOG_ROOT = REPO_ROOT / "backlogProducto"

for p in [FRONTEND_ROOT, RESULTS_ROOT, DOCS_ROOT, BACKLOG_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print("PFI_ROOT:", PFI_ROOT)
print("REPO_ROOT:", REPO_ROOT, "| exists:", REPO_ROOT.exists())
print("FRONTEND_ROOT:", FRONTEND_ROOT)
print("RESULTS_ROOT:", RESULTS_ROOT)


In [ ]:

inputs = {
    "backend_frontend_contract": REPO_ROOT / "docs" / "PF020_PF025_frontend_backend_contract.md",
    "backend_contract": REPO_ROOT / "docs" / "PF020_PF025_spring_backend_contract.md",
    "backend_root": REPO_ROOT / "backend_product_spring",
}
for k, p in inputs.items():
    print(k, "->", p, "| exists:", p.exists())


In [ ]:

def write_file(path: Path, content: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(textwrap.dedent(content).strip() + "\n", encoding="utf-8")
    print("Wrote:", path)

frontend_files = {
  "package.json": "\n{\n  \"name\": \"pfi-rm-lumbar-product-ui\",\n  \"version\": \"0.1.0\",\n  \"private\": true,\n  \"type\": \"module\",\n  \"scripts\": {\n    \"dev\": \"vite --host 0.0.0.0 --port 5173\",\n    \"build\": \"tsc && vite build\",\n    \"preview\": \"vite preview --host 0.0.0.0 --port 4173\"\n  },\n  \"dependencies\": {\n    \"@vitejs/plugin-react\": \"^5.0.0\",\n    \"vite\": \"^7.0.0\",\n    \"typescript\": \"^5.8.0\",\n    \"react\": \"^19.0.0\",\n    \"react-dom\": \"^19.0.0\",\n    \"lucide-react\": \"^0.468.0\"\n  },\n  \"devDependencies\": {\n    \"@types/react\": \"^19.0.0\",\n    \"@types/react-dom\": \"^19.0.0\"\n  }\n}\n",
  "index.html": "\n<!doctype html>\n<html lang=\"es\">\n  <head>\n    <meta charset=\"UTF-8\" />\n    <meta name=\"viewport\" content=\"width=device-width, initial-scale=1.0\" />\n    <title>PFI RM Lumbar - Worklist IA</title>\n  </head>\n  <body>\n    <div id=\"root\"></div>\n    <script type=\"module\" src=\"/src/main.tsx\"></script>\n  </body>\n</html>\n",
  "vite.config.ts": "\nimport { defineConfig } from \"vite\";\nimport react from \"@vitejs/plugin-react\";\n\nexport default defineConfig({\n  plugins: [react()],\n  server: {\n    port: 5173,\n    proxy: {\n      \"/api\": {\n        target: \"http://localhost:8080\",\n        changeOrigin: true\n      }\n    }\n  }\n});\n",
  "tsconfig.json": "\n{\n  \"compilerOptions\": {\n    \"target\": \"ES2020\",\n    \"useDefineForClassFields\": true,\n    \"lib\": [\"DOM\", \"DOM.Iterable\", \"ES2020\"],\n    \"allowJs\": false,\n    \"skipLibCheck\": true,\n    \"esModuleInterop\": true,\n    \"allowSyntheticDefaultImports\": true,\n    \"strict\": true,\n    \"forceConsistentCasingInFileNames\": true,\n    \"module\": \"ESNext\",\n    \"moduleResolution\": \"Node\",\n    \"resolveJsonModule\": true,\n    \"isolatedModules\": true,\n    \"noEmit\": true,\n    \"jsx\": \"react-jsx\"\n  },\n  \"include\": [\"src\"],\n  \"references\": []\n}\n",
  ".env.example": "\nVITE_API_BASE_URL=http://localhost:8080\nVITE_USE_MOCK=false\n",
  "README.md": "\n# Frontend product UI - PFI RM Lumbar\n\nFrontend React/Vite/TypeScript para el MVP técnico de análisis asistido de RM lumbar.\n\n## Alcance\n\nLa interfaz muestra resultados técnicos de inferencia, worklist, overlays/evidencia visual, flags, prioridad y estado de revisión. No presenta diagnóstico clínico autónomo.\n\n## Ejecución\n\n```bash\ncd frontend_product_ui\ncp .env.example .env\nnpm install\nnpm run dev\n```\n\nPor defecto consume Spring Boot en:\n\n```text\nhttp://localhost:8080\n```\n\n## Endpoints esperados\n\n- `GET /api/ai/health`\n- `GET /api/ai/models`\n- `POST /api/ai/pipeline/run`\n- `GET /api/ai/agent/report/{runId}`\n- `PATCH /api/ai/review/{runId}`\n\n## Modo demo\n\nPara usar datos mock sin backend:\n\n```text\nVITE_USE_MOCK=true\n```\n",
  "src/types.ts": "\nexport type ReviewStatus = \"pendiente\" | \"aceptado\" | \"observado\" | \"descartado\";\nexport type Priority = \"baja\" | \"media\" | \"alta\";\n\nexport interface HealthResponse {\n  status: string;\n  service?: string;\n  humanReviewRequired?: boolean;\n}\n\nexport interface AiModel {\n  modelKey: string;\n  plane: \"sagittal\" | \"axial\" | string;\n  datasetKey?: string;\n  checkpointPath?: string;\n  status?: string;\n}\n\nexport interface PipelineRunRequest {\n  plane: \"sagittal\" | \"axial\";\n  modelKey?: string;\n  caseId?: string;\n  inputPath?: string;\n}\n\nexport interface Measurement {\n  classId: number | string;\n  className?: string;\n  foregroundRatio?: number;\n  areaPixels?: number;\n  centroidX?: number;\n  centroidY?: number;\n}\n\nexport interface AgentDecision {\n  priority: Priority;\n  status: string;\n  flags: string[];\n  reasons: string[];\n  recommendedAction: string;\n  humanReviewRequired: boolean;\n  notClinicalDiagnosis: boolean;\n}\n\nexport interface AiRunResponse {\n  runId: string;\n  caseId: string;\n  plane: \"sagittal\" | \"axial\" | string;\n  modelKey: string;\n  createdAt?: string;\n  overlayPath?: string;\n  foregroundRatio?: number;\n  presentClasses?: Array<number | string>;\n  measurements?: Measurement[];\n  agentDecision: AgentDecision;\n  reviewStatus?: ReviewStatus;\n  reviewerNotes?: string;\n}\n\nexport interface ReviewUpdateRequest {\n  status: ReviewStatus;\n  reviewerNotes: string;\n}\n",
  "src/mock/sampleRun.ts": "\nimport type { AiRunResponse, HealthResponse, AiModel } from \"../types\";\n\nexport const mockHealth: HealthResponse = {\n  status: \"ok\",\n  service: \"spring-boot-backend\",\n  humanReviewRequired: true\n};\n\nexport const mockModels: AiModel[] = [\n  {\n    modelKey: \"sagittal_spider\",\n    plane: \"sagittal\",\n    datasetKey: \"spider_sagittal\",\n    status: \"final_checkpoint_consolidated\"\n  },\n  {\n    modelKey: \"axial_t2_alkafri\",\n    plane: \"axial\",\n    datasetKey: \"alkafri_sudirman_axial\",\n    status: \"final_checkpoint_consolidated\"\n  }\n];\n\nexport const mockRuns: AiRunResponse[] = [\n  {\n    runId: \"demo_sagittal_001\",\n    caseId: \"SAG_DEMO_001\",\n    plane: \"sagittal\",\n    modelKey: \"sagittal_spider\",\n    createdAt: \"2026-06-30T22:50:00\",\n    overlayPath: \"/demo/sagittal_overlay.png\",\n    foregroundRatio: 0.146,\n    presentClasses: [1, 2, 3],\n    measurements: [\n      { classId: 1, className: \"vertebra_group\", foregroundRatio: 0.08, areaPixels: 5320 },\n      { classId: 2, className: \"canal\", foregroundRatio: 0.02, areaPixels: 1310 },\n      { classId: 3, className: \"disc_group\", foregroundRatio: 0.04, areaPixels: 2870 }\n    ],\n    agentDecision: {\n      priority: \"media\",\n      status: \"requiere_revision_con_atencion\",\n      flags: [\"componentes_multiples\"],\n      reasons: [\"La salida requiere verificación visual del profesional.\"],\n      recommendedAction: \"Revisar overlay y confirmar segmentación antes de aceptar.\",\n      humanReviewRequired: true,\n      notClinicalDiagnosis: true\n    },\n    reviewStatus: \"pendiente\",\n    reviewerNotes: \"\"\n  },\n  {\n    runId: \"demo_axial_001\",\n    caseId: \"AX_DEMO_001\",\n    plane: \"axial\",\n    modelKey: \"axial_t2_alkafri\",\n    createdAt: \"2026-06-30T22:51:00\",\n    overlayPath: \"/demo/axial_overlay.png\",\n    foregroundRatio: 0.064,\n    presentClasses: [50, 100, 150, 200],\n    measurements: [\n      { classId: 50, className: \"raw_50\", foregroundRatio: 0.02, areaPixels: 1280 },\n      { classId: 100, className: \"raw_100\", foregroundRatio: 0.016, areaPixels: 1030 },\n      { classId: 150, className: \"raw_150\", foregroundRatio: 0.014, areaPixels: 920 },\n      { classId: 200, className: \"raw_200\", foregroundRatio: 0.010, areaPixels: 640 }\n    ],\n    agentDecision: {\n      priority: \"baja\",\n      status: \"listo_para_revision_estandar\",\n      flags: [],\n      reasons: [\"Confianza y proporción de foreground dentro de rangos esperados.\"],\n      recommendedAction: \"Revisión estándar por profesional.\",\n      humanReviewRequired: true,\n      notClinicalDiagnosis: true\n    },\n    reviewStatus: \"pendiente\",\n    reviewerNotes: \"\"\n  }\n];\n",
  "src/api.ts": "\nimport type {\n  AiModel,\n  AiRunResponse,\n  HealthResponse,\n  PipelineRunRequest,\n  ReviewUpdateRequest\n} from \"./types\";\nimport { mockHealth, mockModels, mockRuns } from \"./mock/sampleRun\";\n\nconst API_BASE_URL = import.meta.env.VITE_API_BASE_URL ?? \"http://localhost:8080\";\nconst USE_MOCK = import.meta.env.VITE_USE_MOCK === \"true\";\n\nasync function requestJson<T>(path: string, options?: RequestInit): Promise<T> {\n  const response = await fetch(`${API_BASE_URL}${path}`, {\n    headers: {\n      \"Content-Type\": \"application/json\",\n      ...(options?.headers ?? {})\n    },\n    ...options\n  });\n\n  if (!response.ok) {\n    throw new Error(`HTTP ${response.status} en ${path}`);\n  }\n\n  return response.json() as Promise<T>;\n}\n\nexport const aiApi = {\n  async health(): Promise<HealthResponse> {\n    if (USE_MOCK) return mockHealth;\n    return requestJson<HealthResponse>(\"/api/ai/health\");\n  },\n\n  async models(): Promise<AiModel[]> {\n    if (USE_MOCK) return mockModels;\n    return requestJson<AiModel[]>(\"/api/ai/models\");\n  },\n\n  async runPipeline(payload: PipelineRunRequest): Promise<AiRunResponse> {\n    if (USE_MOCK) {\n      const base = payload.plane === \"axial\" ? mockRuns[1] : mockRuns[0];\n      return {\n        ...base,\n        runId: `mock_${payload.plane}_${Date.now()}`,\n        caseId: payload.caseId || base.caseId,\n        reviewStatus: \"pendiente\",\n        reviewerNotes: \"\"\n      };\n    }\n\n    return requestJson<AiRunResponse>(\"/api/ai/pipeline/run\", {\n      method: \"POST\",\n      body: JSON.stringify(payload)\n    });\n  },\n\n  async getReport(runId: string): Promise<AiRunResponse> {\n    if (USE_MOCK) {\n      return mockRuns.find((r) => r.runId === runId) ?? mockRuns[0];\n    }\n    return requestJson<AiRunResponse>(`/api/ai/agent/report/${runId}`);\n  },\n\n  async updateReview(runId: string, payload: ReviewUpdateRequest): Promise<AiRunResponse> {\n    if (USE_MOCK) {\n      const found = mockRuns.find((r) => r.runId === runId) ?? mockRuns[0];\n      return { ...found, reviewStatus: payload.status, reviewerNotes: payload.reviewerNotes };\n    }\n\n    return requestJson<AiRunResponse>(`/api/ai/review/${runId}`, {\n      method: \"PATCH\",\n      body: JSON.stringify(payload)\n    });\n  }\n};\n",
  "src/main.tsx": "\nimport React from \"react\";\nimport ReactDOM from \"react-dom/client\";\nimport App from \"./App\";\nimport \"./styles.css\";\n\nReactDOM.createRoot(document.getElementById(\"root\") as HTMLElement).render(\n  <React.StrictMode>\n    <App />\n  </React.StrictMode>\n);\n",
  "src/App.tsx": "\nimport { useEffect, useMemo, useState } from \"react\";\nimport { Activity, BrainCircuit, ClipboardCheck, Eye, FileText, RefreshCw, ShieldCheck } from \"lucide-react\";\nimport { aiApi } from \"./api\";\nimport type { AiModel, AiRunResponse, HealthResponse, ReviewStatus } from \"./types\";\nimport { mockRuns } from \"./mock/sampleRun\";\n\nconst reviewStatuses: ReviewStatus[] = [\"pendiente\", \"aceptado\", \"observado\", \"descartado\"];\n\nfunction priorityClass(priority: string) {\n  if (priority === \"alta\") return \"badge badge-high\";\n  if (priority === \"media\") return \"badge badge-medium\";\n  return \"badge badge-low\";\n}\n\nfunction formatPercent(value?: number) {\n  if (value === undefined || value === null) return \"N/D\";\n  return `${(value * 100).toFixed(1)}%`;\n}\n\nexport default function App() {\n  const [health, setHealth] = useState<HealthResponse | null>(null);\n  const [models, setModels] = useState<AiModel[]>([]);\n  const [runs, setRuns] = useState<AiRunResponse[]>(mockRuns);\n  const [selectedRunId, setSelectedRunId] = useState<string>(mockRuns[0].runId);\n  const [loading, setLoading] = useState(false);\n  const [error, setError] = useState<string>(\"\");\n  const selectedRun = useMemo(() => runs.find((run) => run.runId === selectedRunId) ?? runs[0], [runs, selectedRunId]);\n\n  useEffect(() => {\n    Promise.all([aiApi.health(), aiApi.models()])\n      .then(([h, m]) => { setHealth(h); setModels(m); })\n      .catch((err) => setError(err.message));\n  }, []);\n\n  async function executePipeline(plane: \"sagittal\" | \"axial\") {\n    setLoading(true);\n    setError(\"\");\n    try {\n      const modelKey = plane === \"axial\" ? \"axial_t2_alkafri\" : \"sagittal_spider\";\n      const response = await aiApi.runPipeline({ plane, modelKey, caseId: `${plane.toUpperCase()}_PRODUCT_DEMO` });\n      setRuns((current) => [response, ...current]);\n      setSelectedRunId(response.runId);\n    } catch (err) {\n      setError(err instanceof Error ? err.message : \"Error inesperado\");\n    } finally {\n      setLoading(false);\n    }\n  }\n\n  async function updateReview(status: ReviewStatus, reviewerNotes: string) {\n    if (!selectedRun) return;\n    setLoading(true);\n    setError(\"\");\n    try {\n      const updated = await aiApi.updateReview(selectedRun.runId, { status, reviewerNotes });\n      setRuns((current) => current.map((run) => (run.runId === selectedRun.runId ? updated : run)));\n    } catch (err) {\n      setError(err instanceof Error ? err.message : \"Error actualizando revisión\");\n    } finally {\n      setLoading(false);\n    }\n  }\n\n  const stats = {\n    total: runs.length,\n    alta: runs.filter((r) => r.agentDecision.priority === \"alta\").length,\n    media: runs.filter((r) => r.agentDecision.priority === \"media\").length,\n    pendientes: runs.filter((r) => !r.reviewStatus || r.reviewStatus === \"pendiente\").length\n  };\n\n  return (\n    <main className=\"app-shell\">\n      <header className=\"hero\">\n        <div>\n          <p className=\"eyebrow\">PFI · RM lumbar · IA asistiva</p>\n          <h1>Worklist de revisión profesional</h1>\n          <p>Segmentación 2D multiplanar, priorización del agente y revisión humana obligatoria. La salida no constituye diagnóstico clínico autónomo.</p>\n        </div>\n        <div className=\"hero-actions\">\n          <button disabled={loading} onClick={() => executePipeline(\"sagittal\")}><RefreshCw size={16} /> Ejecutar sagital</button>\n          <button disabled={loading} onClick={() => executePipeline(\"axial\")}><RefreshCw size={16} /> Ejecutar axial</button>\n        </div>\n      </header>\n\n      {error && <section className=\"alert\">Error: {error}</section>}\n\n      <section className=\"grid stats-grid\">\n        <MetricCard icon={<Activity />} label=\"Casos/items\" value={stats.total} />\n        <MetricCard icon={<ShieldCheck />} label=\"Revisión pendiente\" value={stats.pendientes} />\n        <MetricCard icon={<BrainCircuit />} label=\"Prioridad alta\" value={stats.alta} />\n        <MetricCard icon={<ClipboardCheck />} label=\"Prioridad media\" value={stats.media} />\n      </section>\n\n      <section className=\"system-strip\">\n        <span>Backend: {health?.status ?? \"sin conexión\"}</span>\n        <span>Modelos registrados: {models.length || \"N/D\"}</span>\n        <span>Human review required: {String(health?.humanReviewRequired ?? true)}</span>\n      </section>\n\n      <section className=\"layout\">\n        <Worklist runs={runs} selectedRunId={selectedRunId} onSelect={setSelectedRunId} />\n        {selectedRun && <CaseDetail run={selectedRun} onReview={updateReview} loading={loading} />}\n      </section>\n    </main>\n  );\n}\n\nfunction MetricCard({ icon, label, value }: { icon: React.ReactNode; label: string; value: number }) {\n  return <article className=\"metric-card\"><div className=\"metric-icon\">{icon}</div><div><p>{label}</p><strong>{value}</strong></div></article>;\n}\n\nfunction Worklist({ runs, selectedRunId, onSelect }: { runs: AiRunResponse[]; selectedRunId: string; onSelect: (runId: string) => void }) {\n  return (\n    <aside className=\"panel worklist\">\n      <div className=\"panel-title\"><FileText size={18} /><h2>Worklist</h2></div>\n      {runs.map((run) => (\n        <button key={run.runId} className={`worklist-item ${run.runId === selectedRunId ? \"active\" : \"\"}`} onClick={() => onSelect(run.runId)}>\n          <div><strong>{run.caseId}</strong><span>{run.plane} · {run.modelKey}</span></div>\n          <span className={priorityClass(run.agentDecision.priority)}>{run.agentDecision.priority}</span>\n        </button>\n      ))}\n    </aside>\n  );\n}\n\nfunction CaseDetail({ run, onReview, loading }: { run: AiRunResponse; onReview: (status: ReviewStatus, notes: string) => Promise<void>; loading: boolean }) {\n  return (\n    <section className=\"panel detail\">\n      <div className=\"detail-header\">\n        <div><p className=\"eyebrow\">Detalle de caso</p><h2>{run.caseId}</h2><p>{run.runId}</p></div>\n        <span className={priorityClass(run.agentDecision.priority)}>prioridad {run.agentDecision.priority}</span>\n      </div>\n      <div className=\"detail-grid\">\n        <EvidencePanel run={run} />\n        <AgentPanel run={run} />\n      </div>\n      <MeasurementsPanel run={run} />\n      <ReviewPanel run={run} onReview={onReview} loading={loading} />\n    </section>\n  );\n}\n\nfunction EvidencePanel({ run }: { run: AiRunResponse }) {\n  return (\n    <article className=\"subpanel evidence\">\n      <div className=\"panel-title\"><Eye size={18} /><h3>Overlay / evidencia visual</h3></div>\n      <div className=\"overlay-placeholder\"><div><strong>{run.plane.toUpperCase()}</strong><span>{run.overlayPath ?? \"overlay pendiente\"}</span></div></div>\n      <div className=\"kv\"><span>Foreground</span><strong>{formatPercent(run.foregroundRatio)}</strong></div>\n      <div className=\"kv\"><span>Clases presentes</span><strong>{run.presentClasses?.join(\", \") || \"N/D\"}</strong></div>\n    </article>\n  );\n}\n\nfunction AgentPanel({ run }: { run: AiRunResponse }) {\n  return (\n    <article className=\"subpanel\">\n      <h3>Agente / orquestador</h3>\n      <p className=\"status-text\">{run.agentDecision.status}</p>\n      <ul className=\"flag-list\">{(run.agentDecision.flags.length ? run.agentDecision.flags : [\"sin_flags\"]).map((flag) => <li key={flag}>{flag}</li>)}</ul>\n      <p>{run.agentDecision.recommendedAction}</p>\n      <div className=\"policy-box\"><strong>Política metodológica</strong><span>Revisión profesional requerida: {String(run.agentDecision.humanReviewRequired)}</span><span>No diagnóstico clínico: {String(run.agentDecision.notClinicalDiagnosis)}</span></div>\n    </article>\n  );\n}\n\nfunction MeasurementsPanel({ run }: { run: AiRunResponse }) {\n  return (\n    <article className=\"subpanel full-width\">\n      <h3>Mediciones mínimas</h3>\n      <div className=\"table\">\n        <div className=\"table-row table-head\"><span>Clase</span><span>Foreground</span><span>Área px</span></div>\n        {(run.measurements ?? []).map((m) => <div className=\"table-row\" key={`${m.classId}-${m.className}`}><span>{m.className ?? m.classId}</span><span>{formatPercent(m.foregroundRatio)}</span><span>{m.areaPixels ?? \"N/D\"}</span></div>)}\n      </div>\n    </article>\n  );\n}\n\nfunction ReviewPanel({ run, onReview, loading }: { run: AiRunResponse; onReview: (status: ReviewStatus, notes: string) => Promise<void>; loading: boolean }) {\n  const [status, setStatus] = useState<ReviewStatus>(run.reviewStatus ?? \"pendiente\");\n  const [notes, setNotes] = useState(run.reviewerNotes ?? \"\");\n  useEffect(() => { setStatus(run.reviewStatus ?? \"pendiente\"); setNotes(run.reviewerNotes ?? \"\"); }, [run.runId, run.reviewStatus, run.reviewerNotes]);\n  return (\n    <article className=\"subpanel full-width review-panel\">\n      <h3>Revisión profesional</h3>\n      <div className=\"review-form\">\n        <label>Estado<select value={status} onChange={(event) => setStatus(event.target.value as ReviewStatus)}>{reviewStatuses.map((item) => <option key={item} value={item}>{item}</option>)}</select></label>\n        <label>Observaciones<textarea value={notes} onChange={(event) => setNotes(event.target.value)} placeholder=\"Registrar observaciones del profesional...\" /></label>\n        <button disabled={loading} onClick={() => onReview(status, notes)}>Guardar revisión</button>\n      </div>\n    </article>\n  );\n}\n",
  "src/styles.css": "\n:root{font-family:Inter,ui-sans-serif,system-ui,-apple-system,BlinkMacSystemFont,\"Segoe UI\",sans-serif;background:#eef2f7;color:#152033}\n*{box-sizing:border-box}body{margin:0}button,select,textarea{font:inherit}button{cursor:pointer}\n.app-shell{max-width:1440px;margin:0 auto;padding:32px}\n.hero{display:flex;justify-content:space-between;gap:24px;align-items:center;padding:32px;border-radius:28px;background:linear-gradient(135deg,#17233b,#2f4f7f);color:white;box-shadow:0 18px 45px rgba(26,42,67,.25)}\n.hero h1{margin:8px 0;font-size:42px}.hero p{max-width:820px;opacity:.92}.eyebrow{text-transform:uppercase;letter-spacing:.08em;font-size:12px;font-weight:700;opacity:.7}\n.hero-actions{display:flex;gap:12px;flex-wrap:wrap}.hero-actions button,.review-form button{border:0;padding:12px 16px;border-radius:14px;background:#fff;color:#17233b;font-weight:700;display:inline-flex;align-items:center;gap:8px}\n.grid{display:grid;gap:16px}.stats-grid{grid-template-columns:repeat(4,minmax(0,1fr));margin:24px 0}\n.metric-card,.panel,.subpanel{background:#fff;border:1px solid #dfe7f2;border-radius:22px;box-shadow:0 12px 30px rgba(19,32,52,.08)}\n.metric-card{display:flex;align-items:center;gap:14px;padding:20px}.metric-card p{margin:0 0 4px;color:#65738a}.metric-card strong{font-size:28px}.metric-icon{width:46px;height:46px;border-radius:16px;background:#edf4ff;display:grid;place-items:center}\n.system-strip{display:flex;gap:16px;flex-wrap:wrap;padding:14px 18px;border-radius:18px;background:#dce9f9;color:#22324f;font-weight:600;margin-bottom:24px}.layout{display:grid;grid-template-columns:360px 1fr;gap:24px}\n.panel{padding:22px}.panel-title{display:flex;align-items:center;gap:10px}.panel-title h2,.panel-title h3,.subpanel h3{margin:0}.worklist{max-height:820px;overflow:auto}\n.worklist-item{width:100%;border:1px solid #dfe7f2;background:#f8fbff;border-radius:16px;padding:14px;margin-top:12px;display:flex;justify-content:space-between;gap:12px;text-align:left}.worklist-item.active{border-color:#3e6fab;background:#edf4ff}.worklist-item strong{display:block;margin-bottom:4px}.worklist-item span{color:#687991;font-size:13px}\n.badge{align-self:flex-start;display:inline-flex;padding:5px 10px;border-radius:999px;font-size:12px;font-weight:800;text-transform:uppercase}.badge-high{background:#ffe4e4;color:#a02222}.badge-medium{background:#fff1cf;color:#8a5a00}.badge-low{background:#dff8ea;color:#1d6b3a}\n.alert{margin:20px 0;padding:14px 18px;border-radius:16px;background:#ffe8e8;color:#971d1d;font-weight:700}.detail-header{display:flex;justify-content:space-between;gap:18px;align-items:flex-start;margin-bottom:18px}.detail-header h2{margin:0}.detail-header p{margin:4px 0;color:#67748a}.detail-grid{display:grid;grid-template-columns:1.15fr 1fr;gap:18px}\n.subpanel{padding:18px}.full-width{margin-top:18px}.overlay-placeholder{height:280px;border-radius:18px;background:radial-gradient(circle at 32% 40%,rgba(94,164,255,.35),transparent 20%),radial-gradient(circle at 60% 58%,rgba(255,179,102,.35),transparent 22%),linear-gradient(135deg,#142039,#39547a);color:white;display:grid;place-items:center;text-align:center;margin:16px 0}.overlay-placeholder strong{display:block;font-size:28px}.overlay-placeholder span{display:block;margin-top:10px;max-width:420px;opacity:.75;word-break:break-word}\n.kv{display:flex;justify-content:space-between;padding:10px 0;border-top:1px solid #e8eef6}.status-text{font-weight:800;color:#2a456d}.flag-list{display:flex;gap:8px;flex-wrap:wrap;list-style:none;padding:0}.flag-list li{padding:6px 10px;border-radius:999px;background:#eef4fb;color:#30415d;font-size:13px}.policy-box{display:grid;gap:6px;padding:14px;border-radius:14px;background:#f5f8fc;margin-top:12px}\n.table{border:1px solid #e2e9f2;border-radius:16px;overflow:hidden}.table-row{display:grid;grid-template-columns:1.2fr 1fr 1fr;padding:12px 14px;border-top:1px solid #e2e9f2}.table-row:first-child{border-top:0}.table-head{background:#f5f8fc;font-weight:800}\n.review-form{display:grid;grid-template-columns:220px 1fr auto;gap:14px;align-items:end}.review-form label{display:grid;gap:6px;color:#51627c;font-weight:700}.review-form select,.review-form textarea{width:100%;border:1px solid #ccd8e8;border-radius:14px;padding:12px;background:#fff}.review-form textarea{min-height:72px;resize:vertical}.review-form button{background:#203657;color:#fff;min-height:46px}\n@media (max-width:980px){.hero,.layout,.detail-grid,.review-form{grid-template-columns:1fr;display:grid}.stats-grid{grid-template-columns:repeat(2,minmax(0,1fr))}}\n"
}
for relative_path, content in frontend_files.items():
    write_file(FRONTEND_ROOT / relative_path, content)

write_file(DOCS_ROOT / "PF026_PF030_frontend_product_ui.md", "\n# PF-026 a PF-030 - Frontend product UI\n\n## Objetivo\n\nConstruir la capa frontend final del MVP técnico para consumir el backend Spring Boot, visualizar worklist, detalle de caso, evidencia visual/overlay, flags, mediciones y estado de revisión profesional.\n\n## Vistas implementadas\n\n1. Dashboard resumen del agente.\n2. Worklist de casos/items.\n3. Detalle de caso.\n4. Vista de overlay/evidencia visual.\n5. Panel de flags, confianza/foreground y acción recomendada.\n6. Estado de revisión profesional.\n7. Formulario de observaciones.\n\n## Contrato backend\n\nEl frontend consume:\n\n- `GET /api/ai/health`\n- `GET /api/ai/models`\n- `POST /api/ai/pipeline/run`\n- `GET /api/ai/agent/report/{runId}`\n- `PATCH /api/ai/review/{runId}`\n\nLa URL base se configura con:\n\n```text\nVITE_API_BASE_URL=http://localhost:8080\n```\n\n## Modo mock\n\nPara demo sin backend:\n\n```text\nVITE_USE_MOCK=true\n```\n\n## Política metodológica\n\nLa interfaz mantiene explícitamente:\n\n- revisión profesional obligatoria,\n- salida técnica asistiva,\n- no diagnóstico clínico autónomo,\n- no reconstrucción 3D real.\n\n## Ejecución\n\n```bash\ncd frontend_product_ui\ncp .env.example .env\nnpm install\nnpm run dev\n```\n")
write_file(DOCS_ROOT / "PF026_PF030_frontend_static_preview.html", "\n<!doctype html>\n<html lang=\"es\">\n<head>\n  <meta charset=\"utf-8\">\n  <title>PF026-PF030 Preview</title>\n  <style>\n    body{font-family:Inter,Arial,sans-serif;background:#eef2f7;margin:0;padding:32px;color:#152033}\n    .hero{background:linear-gradient(135deg,#17233b,#2f4f7f);color:white;padding:32px;border-radius:28px}\n    .grid{display:grid;grid-template-columns:repeat(4,1fr);gap:16px;margin:24px 0}\n    .card,.panel{background:white;border-radius:22px;padding:20px;box-shadow:0 12px 30px rgba(19,32,52,.08)}\n    .layout{display:grid;grid-template-columns:340px 1fr;gap:24px}\n    .badge{display:inline-block;border-radius:999px;padding:6px 10px;background:#fff1cf;color:#8a5a00;font-weight:800}\n    .overlay{height:260px;border-radius:18px;background:linear-gradient(135deg,#142039,#39547a);color:white;display:grid;place-items:center}\n    .warning{background:#dce9f9;border-radius:18px;padding:14px;margin:24px 0;font-weight:700}\n  </style>\n</head>\n<body>\n  <section class=\"hero\">\n    <p>PFI · RM lumbar · IA asistiva</p>\n    <h1>Worklist de revisión profesional</h1>\n    <p>Vista de producto para inferencia 2D multiplanar, priorización y revisión humana obligatoria.</p>\n  </section>\n  <section class=\"grid\">\n    <div class=\"card\"><p>Casos/items</p><h2>2</h2></div>\n    <div class=\"card\"><p>Pendientes</p><h2>2</h2></div>\n    <div class=\"card\"><p>Prioridad alta</p><h2>0</h2></div>\n    <div class=\"card\"><p>Prioridad media</p><h2>1</h2></div>\n  </section>\n  <div class=\"warning\">La salida no constituye diagnóstico clínico autónomo. Requiere revisión profesional.</div>\n  <section class=\"layout\">\n    <aside class=\"panel\">\n      <h2>Worklist</h2>\n      <p><b>SAG_DEMO_001</b> <span class=\"badge\">media</span></p>\n      <p><b>AX_DEMO_001</b> <span class=\"badge\">baja</span></p>\n    </aside>\n    <main class=\"panel\">\n      <h2>Detalle de caso</h2>\n      <div class=\"overlay\">Overlay / evidencia visual</div>\n      <h3>Agente / orquestador</h3>\n      <p>Revisar overlay y confirmar segmentación antes de aceptar.</p>\n      <h3>Revisión profesional</h3>\n      <p>Estado: pendiente · Observaciones: formulario preparado en React.</p>\n    </main>\n  </section>\n</body>\n</html>\n")
write_file(BACKLOG_ROOT / "PF026_PF030_resultados_cierre.md", "\n# Cierre PF-026 a PF-030 - Frontend product UI\n\nEstado: pendiente de validación por ejecución del notebook.\n\n## Resultado esperado\n\nFrontend React/Vite/TypeScript listo para integración con Spring Boot.\n\n## Vistas esperadas\n\n- Dashboard resumen\n- Worklist\n- Detalle de caso\n- Overlay/evidencia visual\n- Flags y acción recomendada\n- Estado de revisión\n- Formulario de observaciones\n\n## Política\n\nLa UI mantiene salida asistiva, revisión profesional requerida y no diagnóstico clínico autónomo.\n")


In [ ]:

required_files = [
    "package.json",
    "index.html",
    "vite.config.ts",
    "tsconfig.json",
    ".env.example",
    "README.md",
    "src/types.ts",
    "src/api.ts",
    "src/main.tsx",
    "src/App.tsx",
    "src/styles.css",
    "src/mock/sampleRun.ts",
]
inventory_rows = []
for rel in required_files:
    p = FRONTEND_ROOT / rel
    inventory_rows.append({
        "relative_path": f"frontend_product_ui/{rel}",
        "exists": p.exists(),
        "size_bytes": p.stat().st_size if p.exists() else 0
    })
inventory_df = pd.DataFrame(inventory_rows)
inventory_csv = RESULTS_ROOT / "PF026_PF030_frontend_files_inventory.csv"
inventory_df.to_csv(inventory_csv, index=False)
display(inventory_df)
print("Inventory:", inventory_csv)


In [ ]:

app_text = (FRONTEND_ROOT / "src" / "App.tsx").read_text(encoding="utf-8")
api_text = (FRONTEND_ROOT / "src" / "api.ts").read_text(encoding="utf-8")
types_text = (FRONTEND_ROOT / "src" / "types.ts").read_text(encoding="utf-8")
readme_text = (FRONTEND_ROOT / "README.md").read_text(encoding="utf-8")

checks = [
    {"check": "frontend_root_exists", "ok": FRONTEND_ROOT.exists(), "detail": str(FRONTEND_ROOT)},
    {"check": "required_files_exist", "ok": bool(inventory_df["exists"].all()), "detail": f"{int(inventory_df['exists'].sum())}/{len(inventory_df)}"},
    {"check": "uses_backend_base_url", "ok": "VITE_API_BASE_URL" in api_text and "/api/ai" in api_text, "detail": "API client consumes Spring Boot"},
    {"check": "has_health_models_pipeline_report_review_calls", "ok": all(s in api_text for s in ["/api/ai/health", "/api/ai/models", "/api/ai/pipeline/run", "/api/ai/agent/report/", "/api/ai/review/"]), "detail": "required backend endpoints referenced"},
    {"check": "has_dashboard_worklist_detail_overlay_review", "ok": all(s in app_text for s in ["Worklist", "Detalle de caso", "Overlay / evidencia visual", "Revisión profesional", "Guardar revisión"]), "detail": "required UI views present"},
    {"check": "has_review_statuses", "ok": all(s in types_text + app_text for s in ["pendiente", "aceptado", "observado", "descartado"]), "detail": "review state flow present"},
    {"check": "preserves_human_review_policy", "ok": "Revisión profesional requerida" in app_text or "humanReviewRequired" in app_text, "detail": "UI displays human review policy"},
    {"check": "mock_fallback_documented", "ok": "VITE_USE_MOCK" in api_text and "VITE_USE_MOCK=true" in readme_text, "detail": "demo fallback available"},
    {"check": "docs_written", "ok": (DOCS_ROOT / "PF026_PF030_frontend_product_ui.md").exists(), "detail": str(DOCS_ROOT / "PF026_PF030_frontend_product_ui.md")},
    {"check": "static_preview_written", "ok": (DOCS_ROOT / "PF026_PF030_frontend_static_preview.html").exists(), "detail": str(DOCS_ROOT / "PF026_PF030_frontend_static_preview.html")},
]
checks_df = pd.DataFrame(checks)
checks_csv = RESULTS_ROOT / "PF026_PF030_checks.csv"
checks_df.to_csv(checks_csv, index=False)
display(checks_df)
print("Checks:", checks_csv)
print("All OK:", bool(checks_df["ok"].all()))


In [ ]:

report = {
    "notebook": "33_PF026_PF030_frontend_product_ui",
    "goal": "generate final product frontend UI consuming Spring Boot backend contract",
    "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
    "inputs": {k: str(v) for k, v in inputs.items()},
    "outputs": {
        "frontend_root": str(FRONTEND_ROOT),
        "frontend_files_inventory_csv": str(inventory_csv),
        "checks_csv": str(checks_csv),
        "docs_repo": str(DOCS_ROOT / "PF026_PF030_frontend_product_ui.md"),
        "static_preview_html": str(DOCS_ROOT / "PF026_PF030_frontend_static_preview.html")
    },
    "summary": {
        "generated_frontend_files": int(inventory_df["exists"].sum()),
        "all_static_checks_ok": bool(checks_df["ok"].all()),
        "views": ["dashboard", "worklist", "case_detail", "overlay_evidence", "agent_flags", "review_status", "review_notes_form"],
        "backend_endpoints_consumed": ["GET /api/ai/health", "GET /api/ai/models", "POST /api/ai/pipeline/run", "GET /api/ai/agent/report/{runId}", "PATCH /api/ai/review/{runId}"]
    },
    "methodological_policy": {
        "frontend_consumes_spring_boot": True,
        "spring_boot_consumes_python_service": True,
        "human_review_required": True,
        "not_clinical_diagnosis": True,
        "not_real_3d_reconstruction": True
    },
    "decision": "PF026_PF030_frontend_ready_for_end_to_end_demo_and_final_validation" if bool(checks_df["ok"].all()) else "PF026_PF030_requires_fixes"
}
report_path = RESULTS_ROOT / "PF026_PF030_report.json"
report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
print("Report:", report_path)
print(json.dumps(report, indent=2, ensure_ascii=False))
